1. What one row means for your lane:
A single row represents a unique (client_id, content_id) pair aggregated over a mid-panel observation window (e.g., March 2026).
2. Which table(s) you will use:
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance.parquet (or the sample parquet file during mechanics testing).
3. Which time window:
Observation Window: Mid-panel month 2026-03 (March 1–March 31, 2026). Outcome Target Window: Subsequent performance change in April 2026.
4. What you'd predict or rank (label or proxy):
Label (is_declining_label): Binary classification (1 if total clicks dropped by >=20% or average position dropped by >=2.0 points in the outcome window; 0 otherwise).
5. One thing you deliberately exclude:
Future performance metrics from April 2026 (like next_month_clicks or trend_direction) inside the feature set to avoid target leakage.

In [24]:
!pip install duckdb pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import duckdb
import pandas as pd

# Load token from your local environment or paste it here directly
# (Remember not to push your actual raw token to a public GitHub repo!)
HF_TOKEN = os.getenv("HF_TOKEN", "hf_myToken")

# Initialize DuckDB connection
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# Data path
DATA_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet"
print("DuckDB connected successfully to Hugging Face!")

DuckDB connected successfully to Hugging Face!


Query 1: Verify the Grain (One row per client_id, content_id)

In [26]:
# Check schema/column names in the sample parquet table
columns_df = con.sql(f"DESCRIBE SELECT * FROM '{DATA_PATH}'").df()
print(columns_df[['column_name', 'column_type']])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [27]:
query_grain = f"""
SELECT 
    client_hash_id, 
    content_hash_id, 
    COUNT(*) as row_count
FROM '{DATA_PATH}'
WHERE month = '2026-03'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5;
"""
res_grain = con.sql(query_grain).df()
print("Duplicate Grain Check (Should be empty):")
print(res_grain)

Duplicate Grain Check (Should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, row_count]
Index: []


Query 2: Slice Row Count & Date Span

In [28]:
query_span = f"""
SELECT 
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT client_hash_id) as total_clients,
    COUNT(DISTINCT content_hash_id) as total_pages,
    COUNT(*) as total_daily_rows
FROM '{DATA_PATH}'
WHERE month = '2026-03';
"""
res_span = con.sql(query_span).df()
print("Date Span and Row Count:")
print(res_span)

Date Span and Row Count:
  min_date max_date  total_clients  total_pages  total_daily_rows
0      NaT      NaT              0            0                 0


Query 3: Availability Check (IS TRUE)

In [29]:
query_availability = f"""
SELECT 
    COUNT(*) as surviving_rows
FROM '{DATA_PATH}'
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
"""
res_avail = con.sql(query_availability).df()
print("Surviving Rows with IS TRUE filter:")
print(res_avail)

Surviving Rows with IS TRUE filter:
   surviving_rows
0               0


Now Lets Build 5 Leakage-Free Features + 1 Intentionally Leaked Feature

In [ ]:
!pip install scikit-learn

In [ ]:
# Aggregate mid-panel month features (March 2026)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

feature_query = f"""
SELECT 
    client_hash_id,
    content_hash_id,
    -- Feature 1: Total Impressions
    SUM(gsc_impressions) AS total_impressions,
    -- Feature 2: Total Clicks
    SUM(gsc_clicks) AS total_clicks,
    -- Feature 3: Average CTR
    CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS avg_ctr,
    -- Feature 4: Average Search Position
    AVG(gsc_avg_position) AS avg_position,
    -- Feature 5: Active Days Ratio
    COUNT(DISTINCT report_date) / 31.0 AS active_days_ratio,
    
    -- THE TRAP: Intentionally leaked label derivative!
    CASE WHEN SUM(gsc_clicks) < 10 THEN 1 ELSE 0 END AS LEAKED_future_decay_signal,
    
    -- Target Label
    CASE WHEN AVG(gsc_avg_position) > 15.0 AND SUM(gsc_clicks) < 5 THEN 1 ELSE 0 END AS is_declining_label
FROM '{DATA_PATH}'
WHERE month = '2026-03' AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) > 100
"""

df_features = con.sql(feature_query).df()
print(f"Feature matrix built with shape: {df_features.shape}")

# Train/Test split grouped by client_hash_id to prevent client leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_features, groups=df_features['client_hash_id']))

train_data = df_features.iloc[train_idx]
test_data = df_features.iloc[test_idx]

y_train = train_data['is_declining_label']
y_test = test_data['is_declining_label']

# 1. WITH LEAKAGE
X_cols_leaked = ['total_impressions', 'total_clicks', 'avg_ctr', 'avg_position', 'active_days_ratio', 'LEAKED_future_decay_signal']
model_leaked = RandomForestClassifier(random_state=42)
model_leaked.fit(train_data[X_cols_leaked], y_train)
preds_leaked = model_leaked.predict(test_data[X_cols_leaked])

print("\n=== WITH LEAKAGE TRAP ===")
print(f"Accuracy: {accuracy_score(y_test, preds_leaked):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_leaked):.4f}")

# 2. HONEST MODEL (LEAKAGE REMOVED)
X_cols_honest = ['total_impressions', 'total_clicks', 'avg_ctr', 'avg_position', 'active_days_ratio']
model_honest = RandomForestClassifier(random_state=42)
model_honest.fit(train_data[X_cols_honest], y_train)
preds_honest = model_honest.predict(test_data[X_cols_honest])

print("\n=== HONEST MODEL (LEAKAGE REMOVED) ===")
print(f"Accuracy: {accuracy_score(y_test, preds_honest):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_honest):.4f}")

ModuleNotFoundError: No module named 'sklearn'

Feature Justifications 
1. total_impressions: Knowable at decision moment because search visibility is recorded historically in March.

2. total_clicks: Knowable at decision moment because user clicks are logged daily in March.

3. avg_ctr: Knowable at decision moment because it is derived strictly from historical clicks/impressions.

4. avg_position: Knowable at decision moment because rank logs exist for March.

5. active_days_ratio: Knowable at decision moment because daily active indexing is known at the end of March.

In [ ]:
# Train/Test split grouped by client_id to prevent client leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_features, groups=df_features['client_id']))

train_data = df_features.iloc[train_idx]
test_data = df_features.iloc[test_idx]

y_train = train_data['is_declining_label']
y_test = test_data['is_declining_label']

# 1. WITH LEAKAGE
X_cols_leaked = ['total_impressions', 'total_clicks', 'avg_ctr', 'avg_position', 'active_days_ratio', 'LEAKED_future_decay_signal']
model_leaked = RandomForestClassifier(random_state=42)
model_leaked.fit(train_data[X_cols_leaked], y_train)
preds_leaked = model_leaked.predict(test_data[X_cols_leaked])

print("=== WITH LEAKAGE TRAP ===")
print(f"Accuracy: {accuracy_score(y_test, preds_leaked):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_leaked):.4f}")

# 2. HONEST MODEL (LEAKAGE REMOVED)
X_cols_honest = ['total_impressions', 'total_clicks', 'avg_ctr', 'avg_position', 'active_days_ratio']
model_honest = RandomForestClassifier(random_state=42)
model_honest.fit(train_data[X_cols_honest], y_train)
preds_honest = model_honest.predict(test_data[X_cols_honest])

print("\n=== HONEST MODEL (LEAKAGE REMOVED) ===")
print(f"Accuracy: {accuracy_score(y_test, preds_honest):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_honest):.4f}")

Named Limitation of this Slice:
This analysis aggregates data over a single 30-day observation window (2026-03). It does not account for macro search seasonality (e.g., holiday or quarterly search spikes) or sudden search engine core algorithm updates occurring outside this mid-panel window.